# Tecnicas de ML para deteccion de fraude y phishing en transacciones financieras digitales

**Notebook narrativo de tesis** — replica la metodologia completa del proyecto sobre Credit Card Fraud Detection y PaySim, presenta los hallazgos consolidados y respalda las conclusiones con figuras y tablas.

**Autor**: Cesar Mendoza Sanchez — Instituto Tecnologico de Morelia

**Estructura**:
1. [Introduccion y motivacion](#1-introduccion-y-motivacion)
2. [Dataset overview](#2-dataset-overview)
3. [EDA comparativo](#3-eda-comparativo)
4. [Descripcion de los 4 modelos individuales](#4-descripcion-de-los-4-modelos-individuales)
5. [Experimentos (sin SMOTE)](#5-experimentos-sin-smote)
6. [Impacto de SMOTE](#6-impacto-de-smote)
7. [Validacion cruzada k-fold](#7-validacion-cruzada-k-fold)
8. [Modelo hibrido](#8-modelo-hibrido)
9. [Conclusiones y trabajo futuro](#9-conclusiones-y-trabajo-futuro)

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures' / 'evaluation'
METRICS_DIR = PROJECT_ROOT / 'reports' / 'metrics'

pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.dpi'] = 100

## 1. Introduccion y motivacion

**TODO**: redactar 2-3 parrafos sobre:
- El problema del fraude transaccional digital en Mexico/LATAM (cifras citables).
- Limitaciones de los sistemas basados en reglas.
- Hipotesis del proyecto: ML supervisado y no supervisado superan a reglas, y la combinacion puede aportar valor incremental.
- Objetivos especificos y alcance.

**Referencias clave** (anadir al cierre):
- Dal Pozzolo et al., 2015 — Credit Card Fraud Detection.
- Lopez-Rojas et al., 2016 — PaySim simulator.
- Awoyemi et al., 2017 — comparativa CC fraud.
- Chawla et al., 2002 — SMOTE.

## 2. Dataset overview

**TODO**: describir ambos datasets:
- **Credit Card Fraud Detection** (Kaggle, anonimizado con PCA): 284,807 transacciones, 0.172% fraude, features V1..V28 + Time + Amount.
- **PaySim** (Kaggle, simulado): 6.36M transacciones, 0.13% fraude, features step, type, amount, oldbalance/newbalance.
- Por que dos datasets: validacion cross-domain de la metodologia.

In [ ]:
# Resumen comparativo de datasets
summary = pd.DataFrame({
    'Dataset': ['Credit Card', 'PaySim'],
    'Filas crudas': [284807, 6362620],
    'Filas tras limpieza': [281918, 6362620],
    'Fraude (n)': [473, 8213],
    'Tasa fraude': ['0.168%', '0.129%'],
    'Features anonimizadas': ['V1..V28 (PCA)', 'No'],
    'Features transaccionales': ['Time, Amount', 'step, type, amount, balances'],
})
summary

## 3. EDA comparativo

**TODO**: ejecutar y mostrar figuras clave de los notebooks 01_eda_creditcard.ipynb y 02_eda_paysim.ipynb:
- Distribucion de la clase fraude.
- Distribucion de monto por clase.
- Tipos de transaccion (PaySim) y tasa de fraude por tipo.
- Correlaciones top con la clase (CC: V17, V14, V12; PaySim: balance_mismatch, is_zero_origin_after).

**Conclusion del EDA**: el fraude en CC requiere informacion anonimizada (PCA), pero en PaySim tiene una huella estructural detectable (vaciado de cuenta + mismatch contable).

## 4. Descripcion de los 4 modelos individuales

**TODO**: una subseccion por modelo con:
- Intuicion del algoritmo.
- Hiperparametros usados y por que.
- Referencia al archivo de configuracion.

1. **RulesBaseline** (`src/fraud_detection/models/rules_baseline.py`): heuristicas derivadas del EDA. 4 reglas por dataset, threshold de activacion configurable.
2. **RandomForest**: bagging de arboles, class_weight balanced.
3. **XGBoost**: boosting secuencial, scale_pos_weight automatico.
4. **IsolationForest**: deteccion de anomalias no supervisada, contamination=auto.

## 5. Experimentos (sin SMOTE)

Resultados consolidados de exp_003 (CC) y exp_004b (PaySim sin isFlaggedFraud) sobre validation set.

**TODO**: cargar JSONs de `reports/metrics/exp_003*__val.json` y `reports/metrics/exp_004b*__val.json` y construir la tabla maestra.

In [ ]:
# Tabla maestra: 4 modelos x 2 datasets sobre validation
results_no_smote = pd.DataFrame([
    {'Modelo': 'RulesBaseline',   'Dataset': 'CreditCard', 'F1': 0.1295, 'AUC': 0.7254, 'P': 0.1250, 'R': 0.1343},
    {'Modelo': 'RandomForest',    'Dataset': 'CreditCard', 'F1': 0.8205, 'AUC': 0.9356, 'P': 0.9600, 'R': 0.7164},
    {'Modelo': 'XGBoost',         'Dataset': 'CreditCard', 'F1': 0.8430, 'AUC': 0.9693, 'P': 0.9444, 'R': 0.7612},
    {'Modelo': 'IsolationForest', 'Dataset': 'CreditCard', 'F1': 0.0431, 'AUC': 0.9259, 'P': 0.0222, 'R': 0.7463},
    {'Modelo': 'RulesBaseline',   'Dataset': 'PaySim',     'F1': 0.0059, 'AUC': 0.9625, 'P': 0.0030, 'R': 1.0000},
    {'Modelo': 'RandomForest',    'Dataset': 'PaySim',     'F1': 0.9980, 'AUC': 1.0000, 'P': 0.9960, 'R': 1.0000},
    {'Modelo': 'XGBoost',         'Dataset': 'PaySim',     'F1': 0.9980, 'AUC': 1.0000, 'P': 0.9960, 'R': 1.0000},
    {'Modelo': 'IsolationForest', 'Dataset': 'PaySim',     'F1': 0.0213, 'AUC': 0.9163, 'P': 0.0108, 'R': 0.7126},
])
results_no_smote

In [ ]:
# Figuras: curvas ROC y matrices de confusion
# TODO: display(Image(FIGURES_DIR / 'exp_003_creditcard_four_models' / 'roc_curve_comparison.png'))
# TODO: display(Image(FIGURES_DIR / 'exp_004b_paysim_four_models_no_flagged' / 'roc_curve_comparison.png'))
pass

## 6. Impacto de SMOTE

Comparativa sin vs con SMOTE en ambos datasets (exp_007 y exp_008).

**TODO**: cargar JSONs de `reports/metrics/exp_007*__val.json` y `reports/metrics/exp_008*__val.json` y construir tabla con deltas.

In [ ]:
# Comparativa SMOTE sobre Credit Card (validation)
smote_cc = pd.DataFrame([
    {'Modelo': 'RandomForest',    'F1 sin': 0.8205, 'F1 con': 0.8571, 'R sin': 0.7164, 'R con': 0.8060},
    {'Modelo': 'XGBoost',         'F1 sin': 0.8430, 'F1 con': 0.8504, 'R sin': 0.7612, 'R con': 0.8060},
    {'Modelo': 'IsolationForest', 'F1 sin': 0.0431, 'F1 con': 0.0514, 'R sin': 0.7463, 'R con': 0.3134},
])
smote_cc['Delta F1'] = smote_cc['F1 con'] - smote_cc['F1 sin']
smote_cc

**Recomendacion final sobre SMOTE** (resumen de docs/baseline_results.md):
- RandomForest en CC: SMOTE aporta valor real (+4.5% F1).
- XGBoost: SMOTE redundante; scale_pos_weight ya compensa.
- IsolationForest: SMOTE contraindicado, distorsiona la nocion de anomalia.
- Datasets saturados (PaySim): SMOTE no aporta, solo duplica tiempo.

## 7. Validacion cruzada k-fold

Reportes generados por `scripts/run_cross_validation.py` (k=5, StratifiedKFold, seed=42).

**TODO**: cargar JSONs de `reports/metrics/cv__*.json` y construir tabla agregada.

In [ ]:
# Resultados k-fold (k=5) sobre modelos ganadores
cv_results = pd.DataFrame([
    {'Dataset': 'CreditCard', 'Modelo': 'XGBoost', 'F1 mean': 0.8576, 'F1 std': 0.0370, 'AUC mean': 0.9798, 'AUC std': 0.0089},
    {'Dataset': 'CreditCard', 'Modelo': 'RandomForest', 'F1 mean': 0.8396, 'F1 std': 0.0390, 'AUC mean': 0.9600, 'AUC std': 0.0098},
    {'Dataset': 'PaySim', 'Modelo': 'XGBoost', 'F1 mean': 0.9957, 'F1 std': 0.0039, 'AUC mean': 0.9990, 'AUC std': 0.0014},
    {'Dataset': 'PaySim', 'Modelo': 'RandomForest', 'F1 mean': 0.9966, 'F1 std': 0.0020, 'AUC mean': 0.9979, 'AUC std': 0.0014},
])
cv_results

**Lectura**: desviaciones estandar pequenas (σF1 ~0.04 en CC, ~0.003 en PaySim) confirman que los resultados del split unico 70/15/15 son reproducibles.

## 8. Modelo hibrido

Implementacion en `src/fraud_detection/models/hybrid.py`: combina XGBoost (supervisado) + IsolationForest (no supervisado) con tres estrategias.

**Estrategias evaluadas** (exp_005, exp_005b, exp_006):
- `or`: union de positivos (maximiza recall).
- `and`: interseccion de positivos (maximiza precision).
- `weighted_voting`: umbral sobre promedio ponderado de probabilidades.

In [ ]:
# Comparativa de estrategias del hibrido en Credit Card (validation)
hybrid_strategies = pd.DataFrame([
    {'Estrategia': 'XGBoost solo (referencia)',           'F1': 0.8430, 'P': 0.9444, 'R': 0.7612, 'TP': 51, 'FP': 3},
    {'Estrategia': 'Hybrid OR',                            'F1': 0.0473, 'P': 0.0244, 'R': 0.8209, 'TP': 55, 'FP': 2203},
    {'Estrategia': 'Hybrid AND',                           'F1': 0.7931, 'P': 0.9388, 'R': 0.6866, 'TP': 46, 'FP': 3},
    {'Estrategia': 'Hybrid weighted_voting (0.7/0.3)',     'F1': 0.8430, 'P': 0.9444, 'R': 0.7612, 'TP': 51, 'FP': 3},
])
hybrid_strategies

**Hallazgo critico**: ninguna estrategia supera al mejor modelo individual (XGBoost) en F1. OR detecta 4 fraudes adicionales pero hereda 2,200 FP del IF; AND descarta fraudes detectables solo por XGBoost; weighted_voting con peso 0.3 del IF queda dominado.

**Conclusion para la tesis**: la complementariedad entre supervisado y no supervisado existe (OR captura fraudes que XGB no), pero el costo en precision hace inviable el hibrido como detector unico. La arquitectura util seria usar OR como filtro de primera linea + segunda etapa de revision o re-ranking; eso queda como trabajo futuro.

## 9. Conclusiones y trabajo futuro

**TODO**: redactar 3-4 parrafos cubriendo:
1. **Validacion de la hipotesis principal**: ML supervisado supera ampliamente a reglas en ambos datasets (F1 +533% en CC, +16,000% en PaySim).
2. **Refutacion de la hipotesis del hibrido naive**: la combinacion OR/AND/weighted_voting no supera al mejor individual.
3. **Recomendacion operacional**: XGBoost con scale_pos_weight automatico es la opcion mas robusta cross-dataset. Para CC, considerar SMOTE solo si se usa RF.
4. **Trabajo futuro**:
   - Evaluacion final sobre el conjunto de test (una sola vez, al cierre del proyecto).
   - Recalibrar `contamination` de IsolationForest a la prevalencia real y reevaluar el hibrido.
   - Re-tuning del baseline de PaySim (`min_rules_to_flag=3` o `=4`).
   - Arquitectura de dos etapas: OR-hibrido como filtro + XGBoost como re-ranker.

**Cierre**: el proyecto demuestra que aplicar ML al problema de fraude transaccional digital aporta valor incremental medible y reproducible, pero subraya que la combinacion ingenua de modelos no garantiza mejora — la arquitectura del sistema importa tanto como la eleccion del algoritmo.